# Glaucoma Detection: 5 CNN Feature Extractors + RF & SVM Classification
**Dataset:** EyePAC-Light v2 (RG vs NRG)

**Pipeline:**
1. Load & preprocess images
2. Extract features using 5 CNN backbones (EfficientNetB3, ResNet50V2, VGG16, InceptionV3, DenseNet121)
3. Compare feature quality (PCA variance, inter-class distance)
4. Classify with Random Forest and SVM on each CNN's features
5. Final comparison table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
import os, gc, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score, accuracy_score
)
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.applications import (
    EfficientNetB3, ResNet50V2, VGG16, InceptionV3, DenseNet121
)
import tensorflow.keras.applications.efficientnet as effnet_utils
import tensorflow.keras.applications.resnet_v2 as resnet_utils
import tensorflow.keras.applications.vgg16 as vgg_utils
import tensorflow.keras.applications.inception_v3 as inception_utils
import tensorflow.keras.applications.densenet as densenet_utils

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Cell 1: Load Dataset

In [ ]:
DATA_ROOT = "/content/drive/MyDrive"
META_CSV  = f"{DATA_ROOT}/eyepac-light-v2-512-jpg/metadata.csv"
IMG_SIZE  = 300          # used for EfficientNet, ResNet, VGG, DenseNet
INCEPTION_SIZE = 299     # InceptionV3 needs 299x299
PER_CLASS = 500          # samples per class; lower if OOM
LABEL_MAP = {"NRG": 0, "RG": 1}

df = pd.read_csv(META_CSV)
df.columns = df.columns.str.strip().str.lower()

if "label" in df.columns:
    df["label"] = df["label"].astype(str).str.strip().str.upper()
    df = df[df["label"].isin(LABEL_MAP)].copy()
    df["y"] = df["label"].map(LABEL_MAP).astype(np.int32)
elif "label_binary" in df.columns:
    df["y"] = df["label_binary"].astype(int)
    df["label"] = df["y"].map({0: "NRG", 1: "RG"})
else:
    raise RuntimeError("Need 'label' or 'label_binary' column")

def build_path(fp, fn):
    return os.path.join(DATA_ROOT, str(fp).lstrip("/"), str(fn).lstrip("/"))

df["image_path"] = df.apply(lambda r: build_path(r["file_path"], r["file_name"]), axis=1)
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"Valid images: {len(df)} | Distribution: {df['label'].value_counts().to_dict()}")

# balanced sample
balanced_df = (
    df.groupby("label", group_keys=False)
      .apply(lambda g: g.sample(n=min(PER_CLASS, len(g)), random_state=SEED)
                        if len(g) >= PER_CLASS
                        else g.sample(n=PER_CLASS, replace=True, random_state=SEED))
      .reset_index(drop=True)
)
print(f"After balancing: {balanced_df['label'].value_counts().to_dict()}")

## Cell 2: Load Raw Images (once, reuse across models)

In [ ]:
# Load images as uint8 [0,255]; each model will preprocess its own copy
raw_images = []
labels = []
paths = []

for _, row in tqdm(balanced_df.iterrows(), total=len(balanced_df), desc="Loading"):
    img = cv2.imread(row["image_path"])
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    raw_images.append(img)
    labels.append(int(row["y"]))
    paths.append(row["image_path"])

raw_images = np.array(raw_images, dtype=np.uint8)   # shape: (N, 300, 300, 3)
y = np.array(labels, dtype=np.int32)
print(f"Loaded: {raw_images.shape} | Labels: {np.unique(y, return_counts=True)}")

# single train/val split — same split for ALL models (fair comparison)
idx = np.arange(len(y))
train_idx, val_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)
y_train, y_val = y[train_idx], y[val_idx]
print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")

## Cell 3: CNN Feature Extractors

| Model | Input Size | Feature Dim | Notes |
|---|---|---|---|
| EfficientNetB3 | 300×300 | 1536 | Compound scaling |
| ResNet50V2 | 300×300 | 2048 | Residual connections |
| VGG16 | 300×300 | 512 | Classic deep CNN |
| InceptionV3 | 299×299 | 2048 | Multi-scale convolutions |
| DenseNet121 | 300×300 | 1024 | Dense connections |

In [ ]:
def build_extractor(backbone_fn, preprocess_fn, input_size):
    """Build a GAP feature extractor from any keras backbone."""
    base = backbone_fn(
        include_top=False,
        weights="imagenet",
        input_shape=(input_size, input_size, 3)
    )
    base.trainable = False
    out = GlobalAveragePooling2D()(base.output)
    model = Model(base.input, out)
    return model, preprocess_fn, input_size


def extract_features(model, preprocess_fn, input_size, images, batch_size=32):
    """Preprocess and extract features in batches."""
    # resize if this model needs different input size
    if images.shape[1] != input_size:
        resized = np.array([
            cv2.resize(img, (input_size, input_size), interpolation=cv2.INTER_AREA)
            for img in images
        ], dtype=np.float32)
    else:
        resized = images.astype(np.float32)

    preprocessed = preprocess_fn(resized)

    feats = []
    for i in range(0, len(preprocessed), batch_size):
        batch = preprocessed[i:i+batch_size]
        feats.append(model.predict(batch, verbose=0))
    return np.vstack(feats)


# define the 5 models
CNN_CONFIGS = [
    ("EfficientNetB3", EfficientNetB3,  effnet_utils.preprocess_input,    IMG_SIZE),
    ("ResNet50V2",     ResNet50V2,       resnet_utils.preprocess_input,    IMG_SIZE),
    ("VGG16",          VGG16,            vgg_utils.preprocess_input,       IMG_SIZE),
    ("InceptionV3",    InceptionV3,      inception_utils.preprocess_input, INCEPTION_SIZE),
    ("DenseNet121",    DenseNet121,      densenet_utils.preprocess_input,  IMG_SIZE),
]

print("Extractor setup done. 5 models ready.")

## Cell 4: Extract Features from All 5 CNNs

In [ ]:
all_features = {}   # name -> {"train": ..., "val": ...}

for name, backbone_fn, preprocess_fn, input_size in CNN_CONFIGS:
    print(f"\n{'='*50}")
    print(f"Extracting: {name} (input: {input_size}x{input_size})")

    extractor, preproc, sz = build_extractor(backbone_fn, preprocess_fn, input_size)

    train_imgs = raw_images[train_idx]
    val_imgs   = raw_images[val_idx]

    feats_train = extract_features(extractor, preproc, sz, train_imgs)
    feats_val   = extract_features(extractor, preproc, sz, val_imgs)

    all_features[name] = {"train": feats_train, "val": feats_val}
    print(f"  Train features: {feats_train.shape} | Val: {feats_val.shape}")

    # free GPU memory
    del extractor
    tf.keras.backend.clear_session()
    gc.collect()

print("\nAll features extracted.")

## Cell 5: Feature Quality Comparison
Compare CNNs using:
- **PCA variance ratio** (how much variance top 50 components capture)
- **Fisher's Criterion** (inter-class vs intra-class separation)
- **Mean cosine distance** between class centroids

In [ ]:
from scipy.spatial.distance import cosine
from numpy.linalg import norm

def feature_quality_metrics(feats, labels, n_components=50):
    scaler = StandardScaler()
    feats_scaled = scaler.fit_transform(feats)

    # PCA variance
    n_comp = min(n_components, feats.shape[1], feats.shape[0] - 1)
    pca = PCA(n_components=n_comp, random_state=SEED)
    pca.fit(feats_scaled)
    pca_var = pca.explained_variance_ratio_.sum()

    # Fisher criterion
    classes = np.unique(labels)
    overall_mean = feats_scaled.mean(axis=0)
    Sb = np.zeros(feats_scaled.shape[1])  # between-class
    Sw = np.zeros(feats_scaled.shape[1])  # within-class
    for c in classes:
        mask = labels == c
        class_mean = feats_scaled[mask].mean(axis=0)
        Sb += mask.sum() * (class_mean - overall_mean) ** 2
        Sw += ((feats_scaled[mask] - class_mean) ** 2).sum(axis=0)
    fisher = (Sb.mean() / (Sw.mean() + 1e-8))

    # cosine distance between class centroids
    c0 = feats_scaled[labels == 0].mean(axis=0)
    c1 = feats_scaled[labels == 1].mean(axis=0)
    centroid_dist = cosine(c0, c1)

    return {"pca_var_50": round(pca_var, 4),
            "fisher": round(fisher, 4),
            "centroid_cosine_dist": round(centroid_dist, 4)}


quality_results = {}
for name, feats in all_features.items():
    q = feature_quality_metrics(feats["train"], y_train)
    quality_results[name] = q
    print(f"{name}: {q}")

quality_df = pd.DataFrame(quality_results).T
print("\nFeature Quality Summary:")
print(quality_df.to_string())

## Cell 6: PCA Visualization (2D) for Each CNN

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
colors = ["#2196F3", "#F44336"]
class_names = ["NRG", "RG"]

for ax, (name, feats) in zip(axes, all_features.items()):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(feats["train"])
    pca2 = PCA(n_components=2, random_state=SEED)
    proj = pca2.fit_transform(scaled)

    for c, (label, color) in enumerate(zip(class_names, colors)):
        mask = y_train == c
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   c=color, label=label, alpha=0.5, s=15, edgecolors='none')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)")

plt.suptitle("PCA 2D Projection — Feature Space per CNN", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("pca_visualization.png", dpi=150, bbox_inches='tight')
plt.show()

## Cell 7: Random Forest Classification on Each CNN's Features

In [ ]:
rf_results = {}

for name, feats in all_features.items():
    print(f"\nRF → {name}")
    X_tr, X_vl = feats["train"], feats["val"]

    # scale
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_vl_s = scaler.transform(X_vl)

    # RF — n_jobs=-1 for speed, class_weight for imbalance robustness
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        class_weight='balanced',
        n_jobs=-1,
        random_state=SEED
    )
    rf.fit(X_tr_s, y_train)

    y_pred = rf.predict(X_vl_s)
    y_prob = rf.predict_proba(X_vl_s)[:, 1]

    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average='macro')
    auc  = roc_auc_score(y_val, y_prob)

    rf_results[name] = {"accuracy": acc, "f1_macro": f1, "auc": auc}
    print(f"  Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

rf_df = pd.DataFrame(rf_results).T
print("\n=== Random Forest Summary ===")
print(rf_df.sort_values('auc', ascending=False).to_string())

## Cell 8: SVM Classification on Each CNN's Features

In [ ]:
svm_results = {}

for name, feats in all_features.items():
    print(f"\nSVM → {name}")
    X_tr, X_vl = feats["train"], feats["val"]

    # scale + PCA (SVM doesn't scale well to 2048-dim raw)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('pca',    PCA(n_components=128, random_state=SEED)),
        ('svm',    SVC(kernel='rbf', C=10, gamma='scale',
                       class_weight='balanced', probability=True,
                       random_state=SEED))
    ])
    pipe.fit(X_tr, y_train)

    y_pred = pipe.predict(X_vl)
    y_prob = pipe.predict_proba(X_vl)[:, 1]

    acc  = accuracy_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred, average='macro')
    auc  = roc_auc_score(y_val, y_prob)

    svm_results[name] = {"accuracy": acc, "f1_macro": f1, "auc": auc}
    print(f"  Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    print(classification_report(y_val, y_pred, target_names=class_names, digits=4))

svm_df = pd.DataFrame(svm_results).T
print("\n=== SVM Summary ===")
print(svm_df.sort_values('auc', ascending=False).to_string())

## Cell 9: Confusion Matrices — Best Setup per CNN

In [ ]:
# pick whichever classifier (RF vs SVM) is better per model by AUC
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for ax, (name, feats) in zip(axes, all_features.items()):
    rf_auc  = rf_results[name]['auc']
    svm_auc = svm_results[name]['auc']
    best_clf_label = "RF" if rf_auc >= svm_auc else "SVM"

    X_tr, X_vl = feats["train"], feats["val"]
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_vl_s = scaler.transform(X_vl)

    if best_clf_label == "RF":
        clf = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                     n_jobs=-1, random_state=SEED)
        clf.fit(X_tr_s, y_train)
        y_pred = clf.predict(X_vl_s)
    else:
        pipe = Pipeline([
            ('pca', PCA(n_components=128, random_state=SEED)),
            ('svm', SVC(kernel='rbf', C=10, gamma='scale',
                        class_weight='balanced', random_state=SEED))
        ])
        pipe.fit(X_tr_s, y_train)
        y_pred = pipe.predict(X_vl_s)

    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{name}\n({best_clf_label}, AUC={max(rf_auc, svm_auc):.3f})",
                 fontsize=10, fontweight='bold')
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.suptitle("Confusion Matrices — Best Classifier per CNN", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.show()

## Cell 10: Final Comparison — All Models, Both Classifiers

In [ ]:
rows = []
for name in all_features:
    rows.append({
        "CNN Backbone": name,
        "Classifier": "RandomForest",
        "Accuracy": rf_results[name]['accuracy'],
        "F1-Macro": rf_results[name]['f1_macro'],
        "AUC": rf_results[name]['auc'],
        "PCA Var (50 comps)": quality_results[name]['pca_var_50'],
        "Fisher Score": quality_results[name]['fisher'],
        "Centroid Distance": quality_results[name]['centroid_cosine_dist'],
    })
    rows.append({
        "CNN Backbone": name,
        "Classifier": "SVM (rbf)",
        "Accuracy": svm_results[name]['accuracy'],
        "F1-Macro": svm_results[name]['f1_macro'],
        "AUC": svm_results[name]['auc'],
        "PCA Var (50 comps)": quality_results[name]['pca_var_50'],
        "Fisher Score": quality_results[name]['fisher'],
        "Centroid Distance": quality_results[name]['centroid_cosine_dist'],
    })

final_df = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("FINAL COMPARISON TABLE")
print("="*80)
print(final_df.to_string(index=False))
print()
print("Best overall:", final_df.iloc[0]['CNN Backbone'], "+", final_df.iloc[0]['Classifier'],
      "| AUC:", final_df.iloc[0]['AUC'])
print("Best feature extractor by Fisher score:",
      max(quality_results, key=lambda k: quality_results[k]['fisher']))

final_df.to_csv("final_comparison.csv", index=False)
print("\nSaved: final_comparison.csv")

## Cell 11: Comparison Bar Charts

In [ ]:
models = list(all_features.keys())
metrics = ['Accuracy', 'F1-Macro', 'AUC']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.arange(len(models))
w = 0.35

for ax, metric in zip(axes, metrics):
    rf_vals  = [rf_results[m][metric.lower().replace('-', '_')] for m in models]
    svm_vals = [svm_results[m][metric.lower().replace('-', '_')] for m in models]

    b1 = ax.bar(x - w/2, rf_vals,  w, label='RandomForest', color='#1976D2', alpha=0.85)
    b2 = ax.bar(x + w/2, svm_vals, w, label='SVM (rbf)',    color='#E53935', alpha=0.85)

    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7)

plt.suptitle("CNN Feature Extractors: RF vs SVM Performance", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("comparison_bars.png", dpi=150, bbox_inches='tight')
plt.show()

## Cell 12: Feature Quality Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
quality_metrics_to_plot = [
    ('pca_var_50',          'PCA Variance (top 50 comps)', 'Higher = more compact features'),
    ('fisher',              'Fisher Score',                'Higher = better class separation'),
    ('centroid_cosine_dist','Centroid Cosine Distance',    'Higher = classes more distinct'),
]

for ax, (key, title, subtitle) in zip(axes, quality_metrics_to_plot):
    vals = [quality_results[m][key] for m in models]
    bars = ax.bar(models, vals, color=['#1976D2','#388E3C','#F57C00','#7B1FA2','#C62828'],
                  alpha=0.85, edgecolor='white')
    ax.set_title(f"{title}\n{subtitle}", fontsize=11, fontweight='bold')
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.suptitle("Feature Quality Comparison Across CNN Backbones", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("feature_quality.png", dpi=150, bbox_inches='tight')
plt.show()

## Cell 13: Grad-CAM on Best CNN (from original project)

In [ ]:
# Last conv layer names per backbone
LAST_CONV = {
    "EfficientNetB3": "top_conv",
    "ResNet50V2":     "post_bn",
    "VGG16":          "block5_conv3",
    "InceptionV3":    "mixed10",
    "DenseNet121":    "bn",
}

# determine best CNN by AUC (taking max of RF and SVM)
best_cnn = max(
    models,
    key=lambda m: max(rf_results[m]['auc'], svm_results[m]['auc'])
)
print(f"Best CNN for Grad-CAM: {best_cnn}")

# rebuild extractor with full model (not just GAP)
best_config = {c[0]: c for c in CNN_CONFIGS}[best_cnn]
_, backbone_fn, preproc_fn, in_sz = best_config

base = backbone_fn(include_top=False, weights='imagenet',
                   input_shape=(in_sz, in_sz, 3))
gradcam_model = base  # use the base directly for Grad-CAM


def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        # use the channel with max mean activation as proxy for 'most active'
        pred_idx = tf.argmax(tf.reduce_mean(preds, axis=[1, 2]), axis=-1)[0]
        class_channel = preds[:, :, :, pred_idx]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(orig_img_rgb, heatmap, alpha=0.4):
    h, w = orig_img_rgb.shape[:2]
    hm = cv2.resize(heatmap, (w, h))
    hm = np.uint8(255 * hm)
    hm_color = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    hm_color = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(orig_img_rgb, 1 - alpha, hm_color, alpha, 0)


# show Grad-CAM on 1 RG and 1 NRG sample
rg_idx_sample  = val_idx[y_val == 1][0]
nrg_idx_sample = val_idx[y_val == 0][0]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for row_i, (sample_idx, label_str) in enumerate([(rg_idx_sample, 'RG'), (nrg_idx_sample, 'NRG')]):
    orig_img = raw_images[sample_idx]  # uint8, 300x300
    if in_sz != IMG_SIZE:
        resized = cv2.resize(orig_img, (in_sz, in_sz))
    else:
        resized = orig_img.copy()

    inp = preproc_fn(resized.astype(np.float32)[np.newaxis, ...])

    try:
        hm = make_gradcam_heatmap(inp, gradcam_model, LAST_CONV[best_cnn])
        overlay = overlay_gradcam(resized, hm)
        axes[row_i][0].imshow(resized)
        axes[row_i][0].set_title(f"Original ({label_str})")
        axes[row_i][1].imshow(overlay)
        axes[row_i][1].set_title(f"Grad-CAM ({best_cnn})")
    except Exception as e:
        print(f"Grad-CAM failed for {best_cnn}: {e}")
        axes[row_i][0].imshow(resized)
        axes[row_i][0].set_title(f"Original ({label_str})")
        axes[row_i][1].axis('off')

    for ax in axes[row_i]:
        ax.axis('off')

plt.suptitle(f"Grad-CAM Visualization: {best_cnn}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("gradcam.png", dpi=150, bbox_inches='tight')
plt.show()

del gradcam_model; gc.collect(); tf.keras.backend.clear_session()

## Cell 14: Summary & Conclusion

In [ ]:
print("\n" + "="*70)
print("EXPERIMENT SUMMARY")
print("="*70)
print(f"Dataset   : EyePAC-Light v2 | Classes: NRG (0), RG (1)")
print(f"Samples   : {len(y)} total | Train: {len(y_train)} | Val: {len(y_val)}")
print(f"CNN Models: EfficientNetB3, ResNet50V2, VGG16, InceptionV3, DenseNet121")
print(f"Classifiers: Random Forest (n=300) | SVM (RBF kernel, C=10, PCA=128)")
print()

print("--- Best Feature Extractor (by Fisher Score) ---")
best_fisher = max(quality_results, key=lambda k: quality_results[k]['fisher'])
print(f"  {best_fisher}: Fisher={quality_results[best_fisher]['fisher']}")

print()
print("--- Top 3 Combinations by AUC ---")
print(final_df[['CNN Backbone','Classifier','AUC','F1-Macro','Accuracy']].head(3).to_string(index=False))

print()
best_row = final_df.iloc[0]
print(f"==> BEST: {best_row['CNN Backbone']} + {best_row['Classifier']}")
print(f"    AUC={best_row['AUC']:.4f} | F1={best_row['F1-Macro']:.4f} | Acc={best_row['Accuracy']:.4f}")